# 01 - Bronze Layer

## Procurement Analytics Pipeline

The Bronze Layer stores raw procurement datasets exactly as received from source systems.

Objectives:

- Read raw CSV files
- Preserve original records
- Add ingestion metadata
- Store raw data in parquet format
- Maintain source-level traceability

## Import Libraries

In [9]:
import os
print(os.environ.get('HADOOP_HOME'))

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

spark = SparkSession.builder \
    .appName('ProcurementAnalytics') \
    .master('local[*]') \
    .getOrCreate()

print('Spark Started')

C:\hadoop
Spark Started


## Load Purchase Orders Dataset

This dataset contains procurement purchase order records.

In [3]:
orders_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/orders_1.csv")

orders_df.show(5, truncate=False)

+---------+---------+-------------+------------------+-------------------+
|po_id    |vendor_id|item_name    |quantity_requested|po_timestamp       |
+---------+---------+-------------+------------------+-------------------+
|PO0000001|V00637   |E-markets    |487               |2025-04-08 15:51:30|
|PO0000002|V01075   |Channels     |326               |2025-05-21 20:44:55|
|PO0000003|V03667   |Models       |24                |2024-05-20 14:26:10|
|PO0000004|V01296   |Architectures|756               |2024-10-18 02:03:33|
|PO0000005|V04012   |Bandwidth    |554               |2025-10-11 07:07:53|
+---------+---------+-------------+------------------+-------------------+
only showing top 5 rows


## Load Invoices Dataset

This dataset contains invoice information received from vendors.

In [4]:
invoices_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/invoices_1.csv")

invoices_df.show(5, truncate=False)

+-----------+---------+-----------------------+-------------------+
|invoice_id |po_id    |invoiced_price_per_unit|invoice_timestamp  |
+-----------+---------+-----------------------+-------------------+
|INV00000001|PO0000665|3865.0                 |2024-08-04 04:12:27|
|INV00000002|PO0001617|282.25                 |2026-02-09 05:45:13|
|INV00000003|PO0002057|4743.39                |2026-01-04 11:32:19|
|INV00000004|PO0001872|4129.53                |2024-06-20 16:52:09|
|INV00000005|PO0003369|3080.2                 |2026-03-29 06:08:10|
+-----------+---------+-----------------------+-------------------+
only showing top 5 rows


## Load Vendors Dataset

This dataset contains vendor master information.

In [5]:
vendors_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/vendors_1.csv")

vendors_df.show(5, truncate=False)

+---------+----------------------+------+-----------+
|vendor_id|vendor_name           |region|risk_rating|
+---------+----------------------+------+-----------+
|V04649   |Dixon-Hughes          |LATAM |Medium     |
|V04163   |Hansen PLC            |EMEA  |High       |
|V03664   |Klein, Chavez and Lane|APAC  |Medium     |
|V03444   |Perez-Harmon          |APAC  |High       |
|V02551   |Taylor Inc            |LATAM |Low        |
+---------+----------------------+------+-----------+
only showing top 5 rows


## Load Vendor Contracts Dataset

This dataset contains negotiated contract prices and validity information.

In [6]:
contracts_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/contracts_1.csv")

contracts_df.show(5, truncate=False)

+-----------+---------+-----------+----------------+----------+
|contract_id|vendor_id|item_name  |negotiated_price|valid_from|
+-----------+---------+-----------+----------------+----------+
|C001960    |V02900   |Communities|888.12          |2025-02-24|
|C001369    |V03799   |Channels   |3717.1          |2026-03-25|
|C004832    |V02993   |Convergence|3054.3          |2024-07-30|
|C004317    |V02067   |Users      |3520.68         |2024-01-20|
|C003307    |V04410   |E-tailers  |3561.33         |2024-12-19|
+-----------+---------+-----------+----------------+----------+
only showing top 5 rows


## Add Ingestion Metadata

An ingestion timestamp is added to every dataset to track when records entered the Bronze Layer.

In [10]:
orders_bronze = orders_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

invoices_bronze = invoices_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

vendors_bronze = vendors_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

contracts_bronze = contracts_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

## Save Bronze Layer Data

Raw datasets are stored in parquet format.

In [13]:
import os

# Create folder manually
os.makedirs('output/bronze', exist_ok=True)

# Convert to pandas and save
orders_bronze.toPandas().to_csv(
    'output/bronze/orders.csv',
    index=False
)

print('Orders Bronze CSV saved successfully')

d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pand

Orders Bronze CSV saved successfully


In [14]:
invoices_bronze.toPandas().to_csv(
    'output/bronze/invoices.csv',
    index=False
)

print('Invoices Bronze CSV saved successfully')

d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pand

Invoices Bronze CSV saved successfully


d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [15]:
vendors_bronze.toPandas().to_csv(
    'output/bronze/vendors.csv',
    index=False
)

print('Vendors Bronze CSV saved successfully')

d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pand

Vendors Bronze CSV saved successfully


In [16]:
contracts_bronze.toPandas().to_csv(
    'output/bronze/contracts.csv',
    index=False
)

print('Contracts Bronze CSV saved successfully')

d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pand

Contracts Bronze CSV saved successfully


## Validation



In [17]:
print(os.listdir('output/bronze'))

['contracts.csv', 'invoices.csv', 'orders.csv', 'vendors.csv']


# Key Learnings

The Bronze Layer serves as the foundation of the Medallion Architecture.

Activities performed:

- Raw procurement datasets were ingested.
- Source data was preserved without modification.
- Ingestion metadata was added.
- Data was stored in CSV format using Pandas export for local execution.
- Dataset traceability was established.

The Bronze Layer ensures that original records remain available for future processing and auditing.

# Conclusion

The Bronze Layer was successfully implemented by ingesting raw procurement datasets and storing them in CSV format using Pandas export for local execution. No transformations were applied at this stage, ensuring that source data integrity is preserved for downstream Silver and Gold layer processing.